The purpose of this notebook is to explore some data from **Strait of Georgia East** (aka SoG East). For most of this notebook, you can read the description and just click play to run the cell.  Occassionally there will be notes that direct you to make some changes.   The overall idea is:
- Review the Area Backscatter Coefficient (ABC) and Center of Mass (CM) to find an interesting event
- Look at the corresponding Mean Volume Backscatter (MVBS) data, and again, look for an interesting period to focus on
- Optional: Create an ONC account and download the raw Sv data to get a very detailed view

The steps in this notebook are:
- Look at the structure of the scalar data file (CM and ABC data)

- Look at 24 hour average ABC and CM
  - Load the ABC and CM data
  - Plot the two time-series for Two Years (2017-Jun-01 to 2019-Jun-01) to help identify some interesting data
  - Optional: Download and plots data from the co-located CTD, Oxygen, and turbidity sensors

- Look at the structure of the MVBS complex data file
- Load and Plot the MVBS data for two months (October 5th to November 15th, 2017, OR for 2018)
- Load the CM data for the functional groups, and plot it over top of the data (November 25th to December 5th, 2017)
- Download the Original Sv data from SoG East using the ONC Client Library, and see how it compares to the MVBS data



In [ ]:
#Define the relative paths of the contents of this repo

from pathlib import Path
MVBS_folder = Path.cwd() / 'MVBS_examples'
scalar_folder = Path.cwd() / 'CM_ABC_averaged_examples'
ADCP_folder = Path.cwd() / 'extra_data'

scalar_filename = scalar_folder / "SoGEast_20150907_20251001_averaged.mat"

### Step 1:
Look at the structure of the ABC and CM time-series file

In [ ]:
import h5py

def print_mat_structure(filename, root="data_avg"):
    """
    Print structure of a MATLAB v7.3 MAT file in a MATLAB-like format.
    """

    def _print_item(name, obj, indent=0):
        pad = "  " * indent

        if isinstance(obj, h5py.Group):
            print(f"{pad}{name.split('/')[-1]}")
            for key in obj.keys():
                _print_item(f"{name}/{key}", obj[key], indent + 1)

        elif isinstance(obj, h5py.Dataset):
            shape = obj.shape
            dtype = obj.dtype
            print(f"{pad}{name.split('/')[-1]} : {dtype} {shape}")

    with h5py.File(filename, "r") as f:
        if root not in f:
            raise ValueError(f"Root '{root}' not found in file")

        _print_item(root, f[root], indent=0)


# Example usage
if __name__ == "__main__":
    print_mat_structure(scalar_filename)


### Step 2:
Look at a few years of ABC and CM data, using the 24 hour average 
- Load the ABC and CM data
- Plot the two time-series to help identify some interesting data

In [ ]:
#Load the CM, and ABC for Strait of Georgia East Node
# Use the "pre-averaged" data to make cleaner plots (use 24 hr average instead of 5 minute average)
import h5py
import numpy as np
import pandas as pd

#To convert MATLAB datenum to python datetime
from datetime import datetime, timedelta

#Look at data from Strait of Georgia East Node
filename = scalar_filename

with h5py.File(filename,'r') as f:
    #Import Daily averaged data
    data = f['data_avg']['hr_24']

    #Load time variable and convert to datetime
    time_MATLAB = np.array(data['time']).squeeze()    
    # The offset for the Unix epoch (1970-01-01) in MATLAB datenum format is 719529
    unix_epoch_offset_matlab = 719529
    time = pd.to_datetime(time_MATLAB - unix_epoch_offset_matlab, unit='D')

    #Get the Time-series data for the 38 kHz channel
    ch_38 = data['ch_38kHz']
    #ch_38 = data['ch_38kHz_fish']
    CM_38 = np.array(ch_38['CM'])
    ABC_38 = np.array(ch_38['ABC'])

    #Get the Time-series data for the 120 kHz channel
    ch_120 = data['ch_120kHz']
    #ch_120 = data['ch_120kHz_krill2']
    CM_120 = np.array(ch_120['CM'])
    ABC_120 = np.array(ch_120['ABC'])

    #Get the device temperature too (if it exists.  Will exist for Saanich and SoG East, but not Folger Deep)
    temperature = np.array(data['temperature'])

In [ ]:
import matplotlib.pyplot as plt

# Lets plot CM, ABC, and Temperature

# Zoom to a date range
start_zoom = datetime(2023, 6, 1)
end_zoom = datetime(2025, 6, 1)

line_dates = [datetime(2024, 1, 15), datetime(2024, 4, 1)]

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 8), sharex=True)


axes[0].plot(time, CM_38[1,:])  # Plot the second column of CM_38 against time
axes[0].plot(time, CM_120[1,:])  # Plot the second column of CM_120 against time
axes[0].set_xlabel('Time')
axes[0].set_ylabel('ABC m^2 / m^-2')
axes[0].set_xlim(start_zoom, end_zoom)
#axes[0].set_ylim(0, 4*10**-4)
axes[0].legend(['38 kHz', '120 kHz'])
axes[0].grid(True)
#Also, plot some red vertical lines to highlight where there is interesting features
#for line_date in line_dates:
#    axes[0].plot( [line_date, line_date], [0, 100],'r--') 

axes[1].plot(time, ABC_38[1,:])  # Plot the second column of CM_38 against time
axes[1].plot(time, ABC_120[1,:])  # Plot the second column of CM_120 against time
axes[1].set_xlabel('Time')
axes[1].set_ylabel('ABC m^2 / m^-2')
axes[1].set_xlim(start_zoom, end_zoom)
axes[1].set_ylim(0, 4*10**-4)
axes[1].legend(['38 kHz', '120 kHz'])
axes[1].grid(True)
#Also, plot some red vertical lines to highlight where there is interesting features
for line_date in line_dates:
    axes[1].plot( [line_date, line_date], [0, 1],'r--') 
    

axes[2].plot(time, temperature[0,:])  # Plot the first column of CM_38 against time
axes[2].set_xlabel('Time')          
axes[2].set_ylabel('Temperature (°C)')
axes[2].set_xlim(start_zoom, end_zoom)
axes[2].set_ylim(8, 11)



plt.show()

### Optional Step - Step 2b: 
Create an Oceans3.0 account, and download a few hours of CTD and O2 data

In [ ]:
#Request data from Oceans 3.0

from onc.onc import ONC # This requires the ONC API Client Library package

# You will require an ONC account and an ONC token to proceed
# To register, visit: https://data.oceannetworks.ca/Registration
#
# Once registered:
# 1) access your Profile from the top-right of https://data.oceannetworks.ca/
# 2) Click the "Web Services API" tab
# 3) Copy your token
ONC_token = '12345678-abcdefgh-9012345-ijklmno' # Add your token here

### Look for available data
Check for different devices deployed at this time at this site

In [ ]:
#Get a list of deployed devices
onc = ONC(ONC_token)

locationCode = 'SEVIP'
deployments = onc.getDeployments({'locationCode':locationCode,'dateFrom':'2022-01-01T00:00:00.000Z','dateTo':'2026-02-01T00:00:00.000Z'})

for deps in deployments:
    print("begin: {}, end: {}, deviceCategoryCode: {}".format(deps['begin'],deps['end'],deps['deviceCategoryCode']))

In [ ]:
#It looks like there is CTD, Oxygen, and Turbidity data

#With the API, you can specify different averaging intervals.  For 2 months of data:
# For unaveraged request, takes about 15 seconds, but only returns ~1 day (100,000 lines).
# For 1 minute averaged request, for 60 days, takes about 8 minutes.
# For 10 minute averaged request, for 60 days, takes about 6 minutes.
# For 60 minute averaged request, for 60 days, takes about 9 seconds. 


#Request 2 years of data, averaged to 1 day intervals
startDate = '2023-06-01T00:00:00.000Z'
endDate = '2025-06-01T00:00:00.000Z'
deviceCategoryCodes = {'CTD','OXYSENSOR','TURBIDITYMETER'}
data_onc = {k: [] for k in deviceCategoryCodes}

for deviceCategoryCode in deviceCategoryCodes:
    data_onc[deviceCategoryCode] = onc.getDirectByLocation(
        {'locationCode':locationCode,
        'deviceCategoryCode':deviceCategoryCode,
        'dateFrom':startDate,
        'dateTo':endDate,
        'resampleType': 'avg', #Optional parameter
        'resamplePeriod': 86400}); #Optional parameter, others include 60, 600, 900, 3600, 86400


In [ ]:
#Optional: you can save the ONC sensor data if you want to re-use it later
import pickle

with open("output/data_CTD.pkl", "wb") as f:
    pickle.dump(data_onc, f)

# Load later
#with open("output/data_CTD.pkl", "rb") as f:
#    data_onc = pickle.load(f)

In [ ]:
#Look at the contents of the CTD data
print(data_onc['CTD']['sensorData'][0].keys())

In [ ]:
#Look at available sensors in the data
for sensors in data_onc['CTD']['sensorData']:
    print(sensors['sensorName'])

In [ ]:
#Look at structure of the data
data_local = data_onc['CTD']['sensorData'][0]['data']
data_local.keys()

In [ ]:
#Extract the data to a pandas data frame
import pandas as pd

print('Sensor data from each Device')
df = pd.DataFrame()
for keys in data_onc.keys(): 
    print("{}:".format(keys))
    for entry in data_onc[keys]['sensorData']:
        print("  {}".format(entry['sensorName']))
        temp_df = pd.DataFrame({
            'time': pd.to_datetime(entry['data']['sampleTimes']),
            entry['sensorName']: entry['data']['values']
        })
        if df.empty:
            df = temp_df
        else:
            df = pd.merge(df, temp_df, on='time', how='outer')

print(df.columns.tolist())

### Now, lets add the Sensor Data to the plots we already have

In [ ]:
import matplotlib.pyplot as plt

# Lets plot CM, ABC, and Temperature

# Zoom to a date range
start_zoom = datetime(2023, 6, 1)
end_zoom = datetime(2025, 6, 1)

line_dates = [datetime(2024, 1, 15), datetime(2024, 4, 1)]

fig, axes = plt.subplots(nrows=6, ncols=1, figsize=(12, 8), sharex=True)


axes[0].plot(time, CM_38[1,:])  # Plot the second column of CM_38 against time
axes[0].plot(time, CM_120[1,:])  # Plot the second column of CM_120 against time
axes[0].set_xlabel('Time')
axes[0].set_ylabel('CM [m]')
axes[0].set_xlim(start_zoom, end_zoom)
#axes[0].set_ylim(0, 4*10**-4)
axes[0].legend(['38 kHz', '120 kHz'])
axes[0].grid(True)

axes[1].plot(time, ABC_38[1,:])  # Plot the second column of CM_38 against time
axes[1].plot(time, ABC_120[1,:])  # Plot the second column of CM_120 against time
axes[1].set_xlabel('Time')
axes[1].set_ylabel('ABC [m^2 / m^-2]')
axes[1].set_xlim(start_zoom, end_zoom)
axes[1].set_ylim(0, 4*10**-4)
axes[1].legend(['38 kHz', '120 kHz'])
axes[1].grid(True)
#Also, plot some red vertical lines to highlight where there is interesting features
for line_date in line_dates:
    axes[1].plot( [line_date, line_date], [0, 1],'r--') 
    

axes[2].plot(time, temperature[0,:])  # Plot the first column of CM_38 against time
axes[2].plot(df['time'], df['Temperature_x'])  # Plot the first column of CM_38 against time
axes[2].set_xlabel('Time')          
axes[2].set_ylabel('Temperature (°C)')
axes[2].set_xlim(start_zoom, end_zoom)
axes[2].set_ylim(8, 11)
axes[2].legend(['AZFP', 'CTD'])

axes[3].plot(df['time'], df['Practical Salinity'])  # Plot the first column of CM_38 against time
axes[3].set_xlabel('Time')          
axes[3].set_ylabel('Salnity (psu)')
axes[3].set_xlim(start_zoom, end_zoom)
# axes[3].set_ylim(8, 11)
axes[3].legend(['CTD'])

axes[4].plot(df['time'], df['Oxygen Concentration Corrected'])  # Plot the first column of CM_38 against time
axes[4].set_xlabel('Time')          
axes[4].set_ylabel('Oxygen Concentration Corrected')
axes[4].set_xlim(start_zoom, end_zoom)
# axes[3].set_ylim(8, 11)
axes[4].legend(['OXYSENSOR'])

axes[5].plot(df['time'], df['Turbidity'])  # Plot the first column of CM_38 against time
axes[5].set_xlabel('Time')          
axes[5].set_ylabel('Turbidity')
axes[5].set_xlim(start_zoom, end_zoom)
# axes[3].set_ylim(8, 11)
axes[5].legend(['TURBIDITY'])


plt.show()

### Step 3: Load the MVBS data products, and plot them



Look at the structure of the MVBS data

In [ ]:
#Get the filenames of the corresponding MVBS data
MVBS_filename = []
 
MVBS_filename.append(MVBS_folder / "StraitOfGeorgiaEast_ASLAZFP55196_20240101T000000_20240201T000000_MVBS_300s_1m.mat")
MVBS_filename.append(MVBS_folder / "StraitOfGeorgiaEast_ASLAZFP55196_20240201T000000_20240301T000000_MVBS_300s_1m.mat")
MVBS_filename.append(MVBS_folder / "StraitOfGeorgiaEast_ASLAZFP55196_20240301T000000_20240309T230000_MVBS_300s_1m.mat")
MVBS_filename.append(MVBS_folder / "StraitOfGeorgiaEast_ASLAZFP55196_20240313T000000_20240401T000000_MVBS_300s_1m.mat")


In [ ]:
import h5py

def print_mat_structure(filename, root="data"):
    """
    Print structure of a MATLAB v7.3 MAT file in a MATLAB-like format.
    """

    def _print_item(name, obj, indent=0):
        pad = "  " * indent

        if isinstance(obj, h5py.Group):
            print(f"{pad}{name.split('/')[-1]}")
            for key in obj.keys():
                _print_item(f"{name}/{key}", obj[key], indent + 1)

        elif isinstance(obj, h5py.Dataset):
            shape = obj.shape
            dtype = obj.dtype
            print(f"{pad}{name.split('/')[-1]} : {dtype} {shape}")

    with h5py.File(filename, "r") as f:
        if root not in f:
            raise ValueError(f"Root '{root}' not found in file")

        _print_item(root, f[root], indent=0)


# Example usage
if __name__ == "__main__":
    print_mat_structure(MVBS_filename[0])


### Step 4
Load and Plot the MVBS data for the highlighted event

First we need to load and concatenate the data from multiple files

In [ ]:
import h5py
import numpy as np
import pandas as pd

#To convert MATLAB datenum to python datetime
from datetime import datetime, timedelta

#Create keys and a dict for the MVBS
channels = ['ch_38kHz','ch_120kHz','ch_200kHz','ch_38kHz_fish','ch_120kHz_krill','ch_120kHz_krill2'] 
MVBS = {k: [] for k in channels}
MVBS_list = {k: [] for k in channels}

#Initialize lists for loading the data from multiple files
time_MATLAB_list = [] #Time variable


for filename in MVBS_filename:
    with h5py.File(filename,'r') as f:
        #Import hourly averaged data
        data = f['data']

        #Load time variable and convert to datetime
        time_MATLAB_list.append(np.array(data['time']).squeeze())    

        #Get the Time-series data for the 38 kHz channel
        MVBS_load = data['MVBS']
        for ch in channels:
            MVBS_list[ch].append(np.array(MVBS_load[ch]))

        #Load the bin depth data. This will be consistent between files
        depth_data = np.array(data['depth_bins'])


#After loading from all files, concatenate the arrays
for ch in channels:
    MVBS[ch] = np.concatenate(MVBS_list[ch], axis=1)

#Also, concatenate the time values, and convert to datetime
# The offset for the Unix epoch (1970-01-01) in MATLAB datenum format is 719529
unix_epoch_offset_matlab = 719529
time_MATLAB = np.concatenate(time_MATLAB_list)
time = pd.to_datetime(time_MATLAB - unix_epoch_offset_matlab, unit='D')

#Delete the list variables to free up memory
del MVBS_list, data


In [ ]:
#There are some gaps in the files, so need to add NaNs in the gaps
dt = pd.Timedelta(seconds=300)

time_full = pd.date_range(
    start=time[0],
    end=time[-1],
    freq=dt
)

# Convert to integer sample numbers
t0 = time_full[0]

# Map timestamps to indices
idx = np.round( (time - t0) / dt ).astype(int)

MVBS_full = {}
for ch in channels:

    Sv = MVBS[ch]
    Sv_full = np.full(
        (Sv.shape[0], len(time_full)),
        np.nan,
        dtype=Sv.dtype
    )
    Sv_full[:, idx] = Sv
    MVBS_full[ch] = Sv_full

missing = np.sum(idx == -1)
print("Unmatched timestamps:", missing)

time = time_full
MVBS = MVBS_full

Now that the data is ready, let's see what it looks like
- First, plot the MVBS data from the 3 frequency channels

In [ ]:
import matplotlib.pyplot as plt

# 1) Plot MVBS for the 3 frequency channels
# 2) Plot the CM on top of the MVBS data

# Zoom to a date range

start_zoom = datetime(2024, 1, 15)
end_zoom = datetime(2024, 4, 1)


fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 10), sharex=True)

# Set background color for all subplots
for ax in axes:
    ax.set_facecolor([0.0, 0.0, 0.2])  # Dark Blue

#Plot each channel 
im_list = []
for ii, ch in enumerate(channels[0:3]):
    im = axes[ii].imshow(MVBS[ch],
                   extent = [time.min(), time.max(), depth_data.max(), depth_data.min()],
                   aspect = 'auto',
                   origin = 'lower',
                   vmin=-70,
                   vmax=-45,
                   cmap='turbo')
    im_list.append(im)  # keep reference for colorbar
    
    axes[ii].set_title(ch)
    axes[ii].set_xlabel('Time')
    axes[ii].set_ylabel('Depth [m]')
    axes[ii].set_ylim(90, 0)
    #Zoom to the days when the bloom in productivity appears to happen from the ABC data
    axes[ii].set_xlim(start_zoom, end_zoom)
    

# Add a single colorbar for all subplots
fig.colorbar(im_list[0], ax=axes, orientation='vertical', fraction=0.05, pad=0.04)

plt.show()

Next, let's plot the data from each acoustically derived functional group
- Fish
- Krill
- General Zooplankton

In [ ]:
import matplotlib.pyplot as plt

# 1) Plot MVBS for the 3 derived channels

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 10), sharex=True)

# Set background color for all subplots
for ax in axes:
    ax.set_facecolor([0.0, 0.0, 0.2])  # Dark Blue

#Plot each channel 
im_list = []
for ii, ch in enumerate(channels[3:]):
    im = axes[ii].imshow(MVBS[ch],
                   extent = [time.min(), time.max(), depth_data.max(), depth_data.min()],
                   aspect = 'auto',
                   origin = 'lower',
                   vmin=-70,
                   vmax=-45,
                   cmap='turbo')
    im_list.append(im)  # keep reference for colorbar
    
    axes[ii].set_title(ch)
    axes[ii].set_xlabel('Time')
    axes[ii].set_ylabel('Depth [m]')
    axes[ii].set_ylim(90, 0)

    #Zoom to the days when the bloom in productivity appears to happen from the ABC data
    axes[ii].set_xlim(start_zoom, end_zoom)

# Add a single colorbar for all subplots
fig.colorbar(im_list[0], ax=axes, orientation='vertical', fraction=0.05, pad=0.04)

plt.show()

### Step 5
Zoom in closer and plot the CM on top

In [ ]:
#Load the 1 hour averaged CM

# scalar_folder = Path.cwd() / 'CM_ABC_examples'
# scalar_filename = scalar_folder / "SoGEast_20150907_20251001.mat"


filename = scalar_filename

with h5py.File(filename,'r') as f:
    #Import Daily averaged data
    #data = f['data']
    data = f['data_avg']['hr_01']

    #Load time variable and convert to datetime
    time_MATLAB = np.array(data['time']).squeeze()    
    # The offset for the Unix epoch (1970-01-01) in MATLAB datenum format is 719529
    unix_epoch_offset_matlab = 719529
    time_CM = pd.to_datetime(time_MATLAB - unix_epoch_offset_matlab, unit='D')

    # Get the CM for the functional groups
    CM_38_fish = np.array(data['ch_38kHz_fish']['CM'])
    CM_123_krill = np.array(data['ch_120kHz_krill']['CM'])
    CM_123_krill2 = np.array(data['ch_120kHz_krill2']['CM'])

In [ ]:
#Plot the MVBS and 1 hour averaged CM

import matplotlib.pyplot as plt

# 1) Plot MVBS for the 3 channels
# 2) Plot the CM on top of the MVBS data

# Zoom to a date range
start_zoom = datetime(2024, 2, 13)
end_zoom = datetime(2024, 2, 21)

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 10), sharex=True)

# Set background color for all subplots
for ax in axes:
    ax.set_facecolor([0.0, 0.0, 0.2])  # Dark Blue

#Plot each channel 
im_list = []
for ii, ch in enumerate(channels[3:6]):
    im = axes[ii].imshow(MVBS[ch],
                   extent = [time.min(), time.max(), depth_data.max(), depth_data.min()],
                   aspect = 'auto',
                   origin = 'lower',
                   vmin=-70,
                   vmax=-45,
                   cmap='turbo')
    im_list.append(im)  # keep reference for colorbar
    
    axes[ii].set_title(ch)
    axes[ii].set_xlabel('Time')
    axes[ii].set_ylabel('Depth [m]')
    axes[ii].set_ylim(160, 0)
    #Zoom to the days when the bloom in productivity appears to happen from the ABC data
    axes[ii].set_xlim(start_zoom, end_zoom)
    
    #Plot the CM on top
    if ii == 0:
        axes[ii].plot(time_CM,CM_38_fish[0],'r--')
    elif ii == 1:
        axes[ii].plot(time_CM,CM_123_krill[0],'r--')
    elif ii == 2:
        axes[ii].plot(time_CM,CM_123_krill2[0],'r--')
    


# Add a single colorbar for all subplots
fig.colorbar(im_list[0], ax=axes, orientation='vertical', fraction=0.05, pad=0.04)

plt.show()

### Optional Step - Step 6: 
Create an Oceans3.0 account, and download a few hours of Sv data

In [ ]:
from datetime import datetime,timezone, timedelta
import numpy as np
import os

from onc.onc import ONC # This requires the ONC API Client Library package

#Request data from Oceans 3.0
# You will require an ONC account and an ONC token to proceed
# To register, visit: https://data.oceannetworks.ca/Registration
#
# Once registered:
# 1) access your Profile from the top-right of https://data.oceannetworks.ca/
# 2) Click the "Web Services API" tab
# 3) Copy your token
ONC_token = '12345678-abcdefgh-9012345-ijklmno' # Add your token here



## Download ONC Data

In [ ]:
#NOTE: These files are very large, so just 8 hours of data can take a while to download

#Initialize the ONC object for retrieving data
onc = ONC(ONC_token)

locationCode = 'SEVIP.E3'
deviceCategoryCode = 'ECHOSOUNDERBIOA'
deviceCode = 'ASLAZFP55196'
start_time = '2024-02-18T00:00:00.000Z'
end_time = '2024-02-18T08:00:00.000Z'
ensemblePeriod = 0

#Download in MAT :
dataProductCode = 'AAPTS' #  ASL Acoustic Profile Timeseries
filters = {
    'locationCode':locationCode,
    'deviceCategoryCode':deviceCategoryCode,
    'dataProductCode':dataProductCode,
    'extension': 'mat',
    #'deviceCode':deviceCode,
    'dateFrom': start_time,
    'dateTo'  : end_time,
    'dpo_ensemblePeriod': ensemblePeriod, # 0 = data not altered (none). All others in s: 20 (30s), 60, 300, 600, 900, 3600
    'token': ONC_token
    }

result = onc.requestDataProduct(filters)
runData = onc.runDataProduct(result['dpRequestId'], waitComplete=True)
print(runData)
result = onc.downloadDataProduct(runData['runIds'][0])

In [ ]:
#This will have saved to a subfolder called "output"

from pathlib import Path

#Print the names of the downloaded files
Sv_folder = Path("output")  

for f in Sv_folder.glob("*.mat"):
    print(f.name)

## Import the Downloaded AZFP Data

In [ ]:
#Load and plot the data
# Plot the Sv left, and MVBS right
from scipy.io import loadmat
import numpy as np
import pandas as pd

#Get the list of all mat-files in the data-directory
mat_path = list(Sv_folder.glob('StraitOfGeorgia*.mat'))

#Initialize some variables
frequency = np.zeros(3)
channels_Sv = []
depth_bins = []

#Go through all MAT files and load
for mat_file in mat_path:
    mat_dict = loadmat(mat_file,simplify_cells=True)
    #print(mat_dict.keys())
    data = mat_dict['Data']
    meta = mat_dict['Meta']
    config = mat_dict['Config']

    #Get list info, and create keys
    if frequency[0] == 0:
        frequency = config['frequency']
        for ch in range(len(frequency)):
            channels_Sv.append("ch_{}kHz".format(int(frequency[ch])))
            depth_bins.append(np.array(meta['depth'] - data['Channel'][ch]['rangeToBinCentre']))

        Sv = {k: [] for k in channels_Sv}
        Sv_list = {k: [] for k in channels_Sv}
        time_MATLAB_list = []
        time_Sv = []

    #Step through each channel in the data    
    for ii,ch in enumerate(channels_Sv):
        time_MATLAB_list.append(np.array(data['time']))
        Sv_list[ch].append(np.array(data['Channel'][ii]['profileData']))
        #print(np.array(data[ii]['vals']).shape)


#After loading from all files, concatenate the arrays
for ch in channels_Sv:
    Sv[ch] = np.concatenate(Sv_list[ch], axis=0)

    #Also, concatenate the time values, and convert to datetime
    # The offset for the Unix epoch (1970-01-01) in MATLAB datenum format is 719529
    unix_epoch_offset_matlab = 719529
    time_MATLAB = np.concatenate(time_MATLAB_list)
    time_Sv = pd.to_datetime(time_MATLAB - unix_epoch_offset_matlab, unit='D')

#Delete the list variables to free up memory
del Sv_list, data, time_MATLAB, time_MATLAB_list





## Plotting with Matplotlib

In [ ]:
#Plot the MVBS and Sv side by side

import matplotlib.pyplot as plt

from datetime import datetime, timedelta

# 1) Plot MVBS for the 3 channels
# 2) Plot Sv for the 3 channels

# Zoom to a date range

#TODO: ADJUST THE ZOOM AS DESIRED
#start_zoom = datetime(2024, 2, 18, 2, 0, 0)
#end_zoom = datetime(2024, 2, 18, 4, 0, 0)
start_zoom = datetime(2024, 2, 18, 0, 0, 0)
end_zoom = datetime(2024, 2, 18, 8, 0, 0)

#Create the figure
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 10), sharex=True)

# Set background color for all subplots
for ax in axes.flatten():
    ax.set_facecolor([0.0, 0.0, 0.2])  # Dark Blue

#Plot each channel of the MVBS data
im_list = []
for jj in range(2):
    for ii, ch in enumerate(channels[0:3]):
        if jj==0:
            #Plot the MVBS data
            im = axes[ii,0].imshow(MVBS[ch],
                    extent = [time.min(), time.max(), depth_data.max(), depth_data.min()],
                    aspect = 'auto',
                    origin = 'lower',
                    vmin=-70,
                    vmax=-45,
                    cmap='turbo')
            
        elif jj==1:
             #Plot the Sv data
             im = axes[ii,1].imshow(Sv[ch].transpose(),
                   extent = [time_Sv.min(), time_Sv.max(), depth_bins[ii].max(), depth_bins[ii].min()],
                   aspect = 'auto',
                   origin = 'lower',
                   vmin=-70,
                   vmax=-45,
                   cmap='turbo')

        im_list.append(im)  # keep reference for colorbar
        
        axes[ii,jj].set_title(ch)
        axes[ii,jj].set_xlabel('Time')
        axes[ii,jj].set_ylabel('Depth [m]')
        axes[ii,jj].set_ylim(165, -5)
        #Zoom to the days when the bloom in productivity appears to happen from the ABC data
        axes[ii,jj].set_xlim(start_zoom, end_zoom)
    

# Add a single colorbar for all subplots
fig.colorbar(im_list[0], ax=axes, orientation='vertical', fraction=0.05, pad=0.04)

plt.show()

### Observations

- The original Sv data is very rich in detail that is lost with the averaged MVBS data
- However, the MVBS data can be use to tell us where to look in the Sv data for interesting features and events